# Eksplorasi Tugas Mata Kuliah Kecerdasan Buatan

Pada tanggal 10 September 2026, mendapatkan tugas pada mata kuliah ini untuk mencari tahu tentang algoritma A* juga Heuristic.

## What & Why?
A* (A-star) merupakan salah satu algoritma terbaik dan populer untuk melakukan pencarian jalur dan pengunjungan tree. Salah satu penggunaan algoritma ini adalah A* terbilang "pintar" dibanding algoritma pencarian jalur atau pengunjungan tree lainnya.

## How it works?
Pada dasarnya, A* menggunakan perhitungan `f(n) = g(n) + h(n)` untuk memilih node tetangga-nya. Pada setiap langkah, A* menggunakan nilai `f(n)` terendah untuk memilih node tujuan.
- `g(n)`: Ini merupakan sebuah fungsi yang menghitung akumulasi banyaknya pergerakan dari titik awal ke titik saat ini.
- `h(n)`: Fungsi ini disebut juga sebagai **Heuristik**. Heuristik merupakan nilai perkiraan jumlah langkah yang akan dibutuhkan untuk mencapai titik tujuan.

## Heuristik
Kita dapat menghitung nilai heuristik dengan dua cara yaitu:
- Mencari Heuristik Eksak
- Mencari Heuristik Aproksimasi

### Heuristik Eksak
Untuk mencari nilai heuristik eksak kita memerlukan banyak waktu komputasi karena harus melakukan *traversal* dari titik saat ini ke titik tujuan. Solusi dari masalah ini adalah menggunakan nilai heuristik aproksimasi sebagai acuan perhitungan pada algoritma A*.

### Heuristik Aproksimasi
Pada umumnya, ada tiga cara perhitungan nilai heuristik aproksimasi.

#### Jarak Mahattan
```python
h = abs(cell.x - goal.x) + abs(cell.y - goal.y)
```
Perhitungan ini menghasilkan nilai absolut perbedaan antara koordinat x dan y saat ini dengan koordinat x dan y tujuan. Perhitungan ini digunakan ketika kita hanya diizinkan bergerak ke empat arah (kanan, kiri, atas, bawah).

#### Jarak Diagonal
```python
dx = abs(cell.x - goal.x)
dy = abs(cell.y - goal.y)

h = D * (dx + dy) + (D2 - 2 * D) * min(dx, dy)
# D  -> Biaya bergerak 1 langkah lurus
# D2 -> Biaya bergerak 1 langkah diagonal: sqrt(D^2 + D^2)
```
Perhitungan ini menghasilkan nilai absolut maksimum dari perbedaan antara koordinat x dan y saat ini dengan koordinat x dan y tujuan. Perhitungan ini digunakan ketika kita bisa bergerak ke delapan arah (kanan, kiri, atas, bawah, dan diagonal).

##### Jarak Euclidian
```python
h = sqrt((cell.x - goal.x)**2 + (cell.y - goal.y)**2)
```
Perhitungan ini menghasilkan nilai jarak Euclidian. Perhitungan ini digunakan ketika kita bisa bebas bergerak ke berbagai arah.



In [259]:
'''
Implementasi algoritma A* untuk path finding dengan terrain berbentuk matriks 2d berukuran 5 * 5
Pada terrain, obstacle ditandai dengan 1 dan path ditandai sebgai 0
'''
row = 5
col = 5
terrain = [
    [0, 0, 0, 1, 0],
    [0, 1, 0, 0, 0],
    [0, 1, 0, 1, 0],
    [0, 1, 0, 1, 1],
    [0, 0, 0, 0, 0],
]

# Deklarasi path yang masih terbuka, sudah dikunjungi, dan path yang benar
open_list = {}
closed_list = {}

def is_valid(x, y):
    return (x >= 0 and x < col) and (y >= 0 and y < row)

def is_unblocked(x, y):
    return terrain[y][x] != 1

def is_destination(src, goal):
    return (src[0] == goal[0]) and (src[1] == goal[1])

def heuristic(src, goal):
    return abs(src[0] - goal[0]) + abs(src[1] - goal[1])

def calculate(src, goal, f, g):
    result = []
    for dx, dy in [(-1, 0), (1, 0), (0, -1), (0, 1)]:
        dest = (src[0]+dx, src[1]+dy)
        
        if not is_valid(*dest):
            continue

        if not is_unblocked(*dest):
            continue

        h = heuristic(dest, goal)
        new_g = g + 1
        new_dest = (*dest, h + new_g, new_g, *src)

        result.append(new_dest)
    
    return result

src = (0, 0)
goal = (4, 4)

def a_star_traverse():
    global open_list, closed_list, path, src, goal

    # Nilai awal open_list adalah titik saat ini
    if len(open_list) == 0:
        open_list[src] = closed_list[src] = (*src, heuristic(src, goal), 0, -1, -1)

    # Lakukan eksplorasi awal
    explore = calculate(src, goal, *open_list[src][2:4])

    # Lakukan perulangan traversal hingga lokasi ditemukan atau seluruh lokasi sudah diperiksa
    while not is_destination(src, goal) and len(explore) != 0:
        # Menambahkan dan/atau update open_list dari hasil eksplorasi
        for coor in explore:
            dest = (coor[0], coor[1])
            dx, dy = dest

            # Menambahkan koordinat ke open_list jika belum terdaftar
            if not dest in open_list:
                open_list[dest] = coor
            else:
                # Update open_list jika ditemukan nilai f lebih kecil
                if open_list[dest][2] > coor[2]:
                    open_list[dest] = coor
        
        # Mencari node dari open_list dengan nilai g tertinggi dan f terendah
        _dest = [*open_list.values()]
        _dest.sort(key=lambda x: x[3], reverse=True) # Pengurutan g tertinggi
        _dest.sort(key=lambda x: x[2]) # Pengurutan f terendah
        dest = _dest[0]

        # Mengembalikan error jika seluruh path sudah dijalankan tapi tidak mendapatkan hasil
        if len(explore) == 0:
            raise ValueError("Path tidak ditemukan!")

        # Mencari node lain jika hasil eksplorasi ke jalur sebelumnya (buntu)
        if dest[:2] in closed_list:
            # Mencari node dari open_list dengan nilai g tertinggi dan f terendah dan tidak terdaftar closed_list (belum dikunjungi)
            _dest = [*open_list.values()]
            _dest.sort(key=lambda x: x[3], reverse=True)
            _dest.sort(key=lambda x: x[2])
            dest = _dest[0]
        
        # Menambahkan ke closed_list jika kunjungan pertama dan menghapus dari open_list
        if not dest in closed_list:
            closed_list[dest[:2]] = dest
            del open_list[dest[:2]]
        
        # Memindahkan ke koordinat baru serta melakukan iterasi eksplorasi
        src = dest[:2]
        explore = calculate(src, goal, *dest[2:4])

try:
    a_star_traverse()
    
    if not goal in closed_list:
        print("Path tidak ditemukan.")
    else:
        # Menyusun alur path terdekat
        path = []
        target = goal
        while target in closed_list:
            path.append(f"({target[0]}, {target[1]})")
            target = closed_list[target][4:]
        
        # Menampilkan alur path
        path.reverse()
        print(" -> ".join(path))
except Exception as e:
    print("Terjadi error saat melakukan traversal.", e)

(0, 0) -> (1, 0) -> (2, 0) -> (2, 1) -> (2, 2) -> (2, 3) -> (2, 4) -> (3, 4) -> (4, 4)
